# Crédito rural (BCB/SICOR) + produção agrícola (IBGE/PAM) em uma tabela

Notebook único. Para cada **município, cultura e ano** (soja, milho e cana-de-açúcar, 2016–2025) monta:

- **IBGE (PAM, tabela 1612):** `area_plantada_ha`, `producao_t`, `rendimento_kg_ha` — coletados pela API do IBGE (precisa de internet);
- **Banco Central (SICOR, custeio):** `credito_nominal_brl` e `area_financiada_ha` — lidos dos arquivos `.json` da pasta `dados`.

**Antes de rodar**

1. Este notebook, a pasta `dados` (com os `.json` do BCB, soltos ou em `dados/bcb/cache`) e a pasta `certs` (com `SectigoPublicServerAuthenticationCADVR36.pem`) devem estar na mesma pasta de trabalho do VS Code.
2. Use **Executar Tudo**. A coleta do IBGE leva alguns minutos (30 consultas).

**Saídas gravadas em `dados/`:** `credito_anual.csv` e `tabela_final.csv`.

## 1. Bibliotecas e configurações

In [14]:
import json
import re
import ssl
from pathlib import Path
from urllib.parse import parse_qs, urlparse

import pandas as pd
import requests as rqs
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Pasta com os .json do BCB (na pasta atual ou na pasta acima)
PASTA_DADOS = next((p for p in (Path("dados"), Path("..") / "dados") if p.is_dir()), Path("dados"))

# Chaves que identificam uma linha da tabela final
CHAVES = ["codigo_municipio", "cultura", "ano"]
ANOS = list(range(2016, 2026))

# ---- IBGE: tabela 1612 (Produção Agrícola Municipal, lavouras temporárias) ----
TABELA_IBGE = 1612
VAR_AREA_PLANTADA_HA = 109      # hectares
VAR_PRODUCAO_T = 214            # toneladas
VAR_RENDIMENTO_KG_HA = 112      # kg/ha
cultura_por_id = {"2713": "soja", "2711": "milho", "2696": "cana_de_acucar"}

# ---- BCB: nome do produto no SICOR -> nome padronizado ----
PRODUTO_PARA_CULTURA = {
    "SOJA": "soja",
    "MILHO": "milho",
    "CANA-DE-AÇUCAR": "cana_de_acucar",
}

## 2. Coleta do IBGE

### 2.1 Conexão segura com o IBGE
O servidor do IBGE entrega a cadeia de certificados incompleta. O adaptador abaixo completa a cadeia com o certificado intermediário local, **mantendo a validação TLS ativa**.

In [15]:
class IBGETLSAdapter(HTTPAdapter):
    """Completa a cadeia do IBGE com o certificado intermediário, sem desativar a validação TLS."""

    def __init__(self, certificado_intermediario, **kwargs):
        self.ssl_context = ssl.create_default_context(cafile=rqs.certs.where())
        # O intermediário deve encadear até uma raiz confiável, sem cadeia parcial.
        self.ssl_context.verify_flags &= ~ssl.VERIFY_X509_PARTIAL_CHAIN
        self.ssl_context.load_verify_locations(cafile=str(certificado_intermediario))
        super().__init__(**kwargs)

    def init_poolmanager(self, connections, maxsize, block=False, **pool_kwargs):
        pool_kwargs["ssl_context"] = self.ssl_context
        return super().init_poolmanager(connections, maxsize, block=block, **pool_kwargs)

    def proxy_manager_for(self, proxy, **proxy_kwargs):
        proxy_kwargs["ssl_context"] = self.ssl_context
        return super().proxy_manager_for(proxy, **proxy_kwargs)

### 2.2 Função de requisição
Faz a consulta com o certificado, **tenta de novo até 5 vezes** (com espera crescente) se a conexão falhar e espera até 15 s para conectar e 120 s para receber os dados.

In [16]:
NOME_CERTIFICADO = "SectigoPublicServerAuthenticationCADVR36.pem"


def localizar_certificado() -> Path:
    for pasta in (Path("certs"), Path("..") / "certs"):
        if (pasta / NOME_CERTIFICADO).is_file():
            return pasta / NOME_CERTIFICADO
    raise FileNotFoundError(
        f"Certificado '{NOME_CERTIFICADO}' não encontrado em 'certs/'. "
        f"Pasta de trabalho atual: {Path.cwd()}"
    )


def requisicao_certificada(url: str, parametros: dict | None = None):
    certificado = localizar_certificado()
    tentativas = Retry(total=5, connect=5, read=3, backoff_factor=2,
                       status_forcelist=[429, 500, 502, 503, 504])
    with rqs.Session() as sessao:
        sessao.mount("https://servicodados.ibge.gov.br/",
                     IBGETLSAdapter(certificado, max_retries=tentativas))
        resposta = sessao.get(url, params=parametros, timeout=(15, 120))
        resposta.raise_for_status()
        return resposta.json()

### 2.3 Função de extração por município
Consulta uma variável da tabela 1612 para **todos os municípios** (`N6`), ano a ano, para soja, milho e cana-de-açúcar.

**Tratamento dos valores (convenção do IBGE):** `-` significa zero e vira `0`; valores só com dígitos viram inteiros; `..`, `...`, `X` e outros textos **não são números** e são descartados. A quantidade de descartes aparece no final de cada coleta.

In [17]:
def extrair_ibge_municipal(codigo_variavel: int, anos=ANOS) -> pd.DataFrame:
    parametros = {
        "localidades": "N6[all]",
        "classificacao": f"81[{','.join(cultura_por_id)}]",
    }
    linhas = []
    descartados = 0

    for ano in anos:
        print(ano, end=" ", flush=True)
        url = (
            "https://servicodados.ibge.gov.br/api/v3/agregados/"
            f"{TABELA_IBGE}/periodos/{ano}/variaveis/{codigo_variavel}"
        )
        resposta = requisicao_certificada(url, parametros)
        if not resposta:
            print(f"\n[aviso] sem dados para a variável {codigo_variavel} em {ano}")
            continue

        variavel = resposta[0]
        for resultado in variavel["resultados"]:
            categorias = resultado["classificacoes"][0]["categoria"]
            for codigo_cultura, nome_cultura in categorias.items():
                for serie in resultado["series"]:
                    for ano_serie, valor in serie["serie"].items():
                        if valor == "-":
                            valor = 0
                        elif valor is None or not str(valor).isdecimal():
                            descartados += 1
                            continue
                        linhas.append({
                            "codigo_cultura": codigo_cultura,
                            "cultura": nome_cultura,
                            "codigo_municipio": serie["localidade"]["id"],
                            "municipio": serie["localidade"]["nome"],
                            "ano": int(ano_serie),
                            "variavel": variavel["variavel"],
                            "unidade": variavel["unidade"],
                            "valor": int(valor),
                        })

    df = pd.DataFrame(linhas)
    print(f"\n-> {len(df):,} linhas | {descartados:,} valores não numéricos descartados")
    return df

### 2.4 Área plantada (ha), produção (t) e rendimento (kg/ha) por município

In [18]:
df_area_municipal = extrair_ibge_municipal(VAR_AREA_PLANTADA_HA)

2016 2017 2018 2019 2020 2021 2022 2023 2024 2025 
-> 164,763 linhas | 2,157 valores não numéricos descartados


In [19]:
df_prod_municipal = extrair_ibge_municipal(VAR_PRODUCAO_T)

2016 2017 2018 2019 2020 2021 2022 2023 2024 2025 
-> 164,763 linhas | 2,157 valores não numéricos descartados


In [20]:
df_rendimento_municipal = extrair_ibge_municipal(VAR_RENDIMENTO_KG_HA)

2016 2017 2018 2019 2020 2021 2022 2023 2024 2025 
-> 164,763 linhas | 2,157 valores não numéricos descartados


In [21]:
# Confere se as três coletas trouxeram dados antes de seguir
for nome, d in {"área": df_area_municipal, "produção": df_prod_municipal,
                "rendimento": df_rendimento_municipal}.items():
    if d.empty:
        raise RuntimeError(f"A coleta de {nome} do IBGE voltou vazia.")
    print(f"{nome}: {len(d):,} linhas | anos {d['ano'].min()}–{d['ano'].max()} | "
          f"{d['codigo_municipio'].nunique():,} municípios")

área: 164,763 linhas | anos 2016–2025 | 5,523 municípios
produção: 164,763 linhas | anos 2016–2025 | 5,523 municípios
rendimento: 164,763 linhas | anos 2016–2025 | 5,523 municípios


### 2.5 Uma tabela do IBGE
Junta as três medidas por município, cultura e ano (junção *outer*: nenhum município some se faltar uma das medidas).

In [22]:
medidas = {
    "area_plantada_ha": df_area_municipal,
    "producao_t": df_prod_municipal,
    "rendimento_kg_ha": df_rendimento_municipal,
}

ibge = None
nomes = []

for nome, df in medidas.items():
    df = df.copy()
    df["cultura"] = df["codigo_cultura"].astype(str).map(cultura_por_id)
    df["codigo_municipio"] = df["codigo_municipio"].astype("string").str.strip()
    df["ano"] = df["ano"].astype("int64")

    # Falha cedo se algo estiver fora do esperado, em vez de gerar uma tabela errada
    if df["cultura"].isna().any():
        desconhecidos = df.loc[df["cultura"].isna(), "codigo_cultura"].unique()
        raise ValueError(f"{nome}: códigos de cultura fora de cultura_por_id: {desconhecidos}")
    duplicadas = df.duplicated(CHAVES).sum()
    if duplicadas:
        raise ValueError(f"{nome}: {duplicadas} linhas repetidas para {CHAVES}; o merge multiplicaria linhas.")

    nomes.append(df[["codigo_municipio", "municipio"]])
    parte = df[CHAVES + ["valor"]].rename(columns={"valor": nome})
    ibge = parte if ibge is None else ibge.merge(parte, on=CHAVES, how="outer")

nomes = pd.concat(nomes).drop_duplicates("codigo_municipio")

print(f"Tabela IBGE: {len(ibge):,} linhas | anos {ibge['ano'].min()}–{ibge['ano'].max()} | "
      f"{ibge['codigo_municipio'].nunique():,} municípios")
ibge.head()

Tabela IBGE: 164,763 linhas | anos 2016–2025 | 5,523 municípios


,codigo_municipio,cultura,ano,area_plantada_ha,producao_t,rendimento_kg_ha
0,1100015,cana_de_acucar,2016,237,15225,73551
1,1100015,cana_de_acucar,2017,31,1085,35000
2,1100015,cana_de_acucar,2018,0,0,0
3,1100015,cana_de_acucar,2019,0,0,0
4,1100015,cana_de_acucar,2020,10,300,30000


## 3. Crédito rural do Banco Central (SICOR)

Cada `.json` da pasta `dados` guarda a resposta de uma consulta `CusteioMunicipioProduto` (um ano, um mês e um produto) com a URL usada. A função abaixo:

1. localiza a pasta do cache (arquivos soltos em `dados/` ou em `dados/bcb/cache`);
2. lê apenas as consultas completas de **ano + mês + produto** (ignora consultas de teste e qualquer outro `.json`);
3. recusa respostas possivelmente truncadas (quantidade de linhas igual ao limite `$top`);
4. avisa se faltar algum mês, ano ou cultura.

In [23]:
RE_FILTRO = re.compile(
    r"AnoEmissao eq '(\d{4})' and MesEmissao eq '(\d{2})' "
    r"and Atividade eq '1' and nomeProduto eq '\"(.+)\"'"
)


def localizar_cache(raiz: Path) -> Path:
    """Procura a pasta do cache, aceitando os .json soltos em 'dados' ou em subpastas."""
    candidatos = [raiz / "bcb" / "cache", raiz / "cache", raiz, Path("bcb") / "cache"]
    for pasta in candidatos:
        if pasta.is_dir() and any(pasta.glob("*.json")):
            return pasta
    raise FileNotFoundError()


def carregar_sicor(pasta_cache: Path, anos=ANOS) -> pd.DataFrame:
    """Lê o cache e devolve os registros brutos de custeio (uma linha por registro do SICOR)."""
    por_consulta = {}   # (ano, mes, produto) -> DataFrame
    ignorados = 0
    truncados = []

    for arquivo in sorted(pasta_cache.glob("*.json")):
        try:
            with open(arquivo, encoding="utf-8") as f:
                cache = json.load(f)
        except ValueError:
            ignorados += 1
            continue
        if not isinstance(cache, dict) or "url" not in cache or "dados" not in cache:
            ignorados += 1       # não é um arquivo de cache do SICOR
            continue

        consulta = parse_qs(urlparse(cache["url"]).query)
        m = RE_FILTRO.fullmatch(consulta.get("$filter", [""])[0])
        if m is None:            # consulta de teste (contains / top=10)
            ignorados += 1
            continue

        ano, mes, produto = m.groups()
        valores = cache["dados"].get("value", [])
        if len(valores) >= int(consulta["$top"][0]):
            truncados.append((ano, mes, produto))

        # Se a mesma consulta aparecer em dois arquivos, mantém o mais completo
        chave = (ano, mes, produto)
        if chave not in por_consulta or len(valores) > len(por_consulta[chave]):
            por_consulta[chave] = pd.DataFrame(valores)

    if truncados:
        raise ValueError(f"Respostas possivelmente truncadas (atingiram o limite $top): {truncados}")

    esperadas = {(str(a), f"{m:02d}", p) for a in anos for m in range(1, 13) for p in PRODUTO_PARA_CULTURA}
    ausentes = sorted(esperadas - set(por_consulta))
    if ausentes:
        print(f"ATENÇÃO: {len(ausentes)} combinações ano/mês/produto sem cache. Exemplos: {ausentes[:5]}")

    print(f"Cache: {len(por_consulta)} consultas usadas, {ignorados} arquivos ignorados.")
    return pd.concat(por_consulta.values(), ignore_index=True)


pasta_cache = localizar_cache(PASTA_DADOS)
sicor = carregar_sicor(pasta_cache)
print(f"{len(sicor):,} registros do SICOR")

ATENÇÃO: 2 combinações ano/mês/produto sem cache. Exemplos: [('2020', '03', 'SOJA'), ('2023', '08', 'CANA-DE-AÇUCAR')]
Cache: 358 consultas usadas, 6 arquivos ignorados.
831,930 registros do SICOR


In [24]:
sicor = sicor.copy()
sicor["cultura"] = sicor["nomeProduto"].str.strip('"').map(PRODUTO_PARA_CULTURA)
sicor["ano"] = sicor["AnoEmissao"].astype("int64")
sicor["codigo_municipio"] = sicor["codIbge"].astype("string").str.strip()

assert sicor["cultura"].notna().all(), "Há produtos do SICOR fora de PRODUTO_PARA_CULTURA"

# O código do município precisa ter 7 dígitos para casar com o do IBGE
valido = sicor["codigo_municipio"].str.fullmatch(r"\d{7}").fillna(False).astype(bool)
if (~valido).any():
    print(f"Registros sem codIbge válido descartados: {(~valido).sum()} "
          f"(R$ {sicor.loc[~valido, 'VlCusteio'].sum():,.2f})")

credito_anual = (
    sicor[valido]
    .groupby(CHAVES, as_index=False)
    .agg(credito_nominal_brl=("VlCusteio", "sum"), area_financiada_ha=("AreaCusteio", "sum"))
    .round({"credito_nominal_brl": 2, "area_financiada_ha": 2})
)
credito_anual["codigo_municipio"] = credito_anual["codigo_municipio"].astype("string")

PASTA_DADOS.mkdir(exist_ok=True)
credito_anual.to_csv(PASTA_DADOS / "credito_anual.csv", index=False)

print(f"credito_anual: {len(credito_anual):,} linhas | {credito_anual['codigo_municipio'].nunique():,} municípios")
credito_anual.groupby("cultura")["credito_nominal_brl"].sum().div(1e9).round(1).rename("R$ bilhões (nominal, 2016–2025)")

Registros sem codIbge válido descartados: 1 (R$ 1,199,128.00)
credito_anual: 53,999 linhas | 3,736 municípios


cultura
cana_de_acucar     38.6
milho             164.7
soja              427.0
Name: R$ bilhões (nominal, 2016–2025), dtype: float64

## 4. Tabela final (IBGE + crédito)

Junção **outer** nas chaves `codigo_municipio`, `cultura` e `ano`. A coluna `cobertura` mostra a origem de cada linha:

| `cobertura` | Significado |
|---|---|
| `ambos` | há dados do IBGE e crédito |
| `so_ibge` | há produção/área no IBGE, mas nenhum crédito de custeio registrado |
| `so_credito` | há crédito, mas o IBGE não informa a cultura naquele município e ano |

Linhas `so_ibge` **não** têm crédito zero: o dado de crédito simplesmente não existe nelas.

In [25]:
credito = credito_anual[CHAVES + ["credito_nominal_brl", "area_financiada_ha"]].copy()

tabela = ibge.merge(credito, on=CHAVES, how="outer", indicator="cobertura")
tabela["cobertura"] = tabela["cobertura"].astype(str).map(
    {"both": "ambos", "left_only": "so_ibge", "right_only": "so_credito"}
)

# Nome do município (vem do IBGE) e localização
tabela = tabela.merge(nomes, on="codigo_municipio", how="left")
tabela["codigo_uf_ibge"] = tabela["codigo_municipio"].str[:2]

UF_POR_CODIGO = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA", "16": "AP", "17": "TO",
    "21": "MA", "22": "PI", "23": "CE", "24": "RN", "25": "PB", "26": "PE", "27": "AL", "28": "SE", "29": "BA",
    "31": "MG", "32": "ES", "33": "RJ", "35": "SP",
    "41": "PR", "42": "SC", "43": "RS",
    "50": "MS", "51": "MT", "52": "GO", "53": "DF",
}
REGIAO_POR_DIGITO = {"1": "Norte", "2": "Nordeste", "3": "Sudeste", "4": "Sul", "5": "Centro-Oeste"}

tabela["uf"] = tabela["codigo_uf_ibge"].map(UF_POR_CODIGO)
tabela["regiao"] = tabela["codigo_municipio"].str[0].map(REGIAO_POR_DIGITO)

primeiras = ["codigo_municipio", "municipio", "uf", "regiao", "codigo_uf_ibge", "cultura", "ano", "cobertura"]
resto = [c for c in tabela.columns if c not in primeiras]
tabela = tabela[primeiras + resto].sort_values(CHAVES).reset_index(drop=True)

# Verificações finais
assert not tabela.duplicated(CHAVES).any(), "Há linhas repetidas na tabela final"
assert tabela["cobertura"].notna().all()

tabela.to_csv(PASTA_DADOS / "tabela_final.csv", index=False, encoding="utf-8-sig")
print(f"tabela: {len(tabela):,} linhas x {tabela.shape[1]} colunas  ->  {PASTA_DADOS / 'tabela_final.csv'}")
tabela.head()

tabela: 164,775 linhas x 13 colunas  ->  dados\tabela_final.csv


,codigo_municipio,municipio,uf,regiao,codigo_uf_ibge,cultura,ano,cobertura,area_plantada_ha,producao_t,rendimento_kg_ha,credito_nominal_brl,area_financiada_ha
0,1100015,Alta Floresta D'Oeste - RO,RO,Norte,11,cana_de_acucar,2016,so_ibge,237.0,15225.0,73551.0,NaN,NaN
1,1100015,Alta Floresta D'Oeste - RO,RO,Norte,11,cana_de_acucar,2017,so_ibge,31.0,1085.0,35000.0,NaN,NaN
2,1100015,Alta Floresta D'Oeste - RO,RO,Norte,11,cana_de_acucar,2018,so_ibge,0.0,0.0,0.0,NaN,NaN
3,1100015,Alta Floresta D'Oeste - RO,RO,Norte,11,cana_de_acucar,2019,so_ibge,0.0,0.0,0.0,NaN,NaN
4,1100015,Alta Floresta D'Oeste - RO,RO,Norte,11,cana_de_acucar,2020,so_ibge,10.0,300.0,30000.0,NaN,NaN


## 5. Conferência da cobertura

In [26]:
resumo = tabela.pivot_table(index="cultura", columns="cobertura", values="ano", aggfunc="count", fill_value=0)
display(resumo)

sem_nome = tabela["municipio"].isna().sum()
print(f"Linhas sem nome de município (código só existe no crédito): {sem_nome:,}")

# O total de crédito não pode mudar com o merge
assert abs(tabela["credito_nominal_brl"].sum() - credito_anual["credito_nominal_brl"].sum()) < 1, \
    "O total de crédito mudou após o merge"
print("Total de crédito preservado após o merge: OK")

cobertura,ambos,so_credito,so_ibge
cultura,,,
cana_de_acucar,7481,2,47440
milho,24242,7,30679
soja,22264,3,32657


Linhas sem nome de município (código só existe no crédito): 9
Total de crédito preservado após o merge: OK
